# Mini Project: Recommendation Engines

Recommendation engines are algorithms designed to provide personalized suggestions or recommendations to users. These systems analyze user behavior, preferences, and interactions with items (products, movies, music, articles, etc.) to predict and offer items that users are likely to be interested in. Recommendation engines play a crucial role in enhancing user experience, driving engagement, and increasing conversion rates in various applications, including e-commerce, entertainment, content platforms, and more.

There are generally two approaches taken in collaborative filtering and content-based recommendation engines:

**1. Collaborative Filtering:**
Collaborative Filtering is a popular approach to building recommendation systems that leverages the collective behavior of users to make personalized recommendations. It is based on the idea that users who have agreed in the past will likely agree in the future. There are two main types of collaborative filtering:

- **User-based Collaborative Filtering:** This method finds users similar to the target user based on their past interactions (e.g., ratings or purchases). It then recommends items that similar users have liked but the target user has not interacted with yet.

- **Item-based Collaborative Filtering:** In this approach, the system identifies similar items based on user interactions. It recommends items that are similar to the ones the target user has already liked or interacted with.

Collaborative filtering does not require any explicit information about items but relies on the similarity between users or items. It is effective in capturing complex patterns and can provide serendipitous recommendations. However, it suffers from the cold-start problem (i.e., difficulty in recommending to new users or items with no interactions) and scalability challenges in large datasets.

**2. Content-Based Recommendation:**
Content-based recommendation is an alternative approach to building recommendation systems that focuses on the attributes or features of items and users. It leverages the characteristics of items to make recommendations. The key steps involved in content-based recommendation are:

- **Feature Extraction:** For each item, relevant features are extracted. For movies, these features could be genre, director, actors, and plot summary.

- **User Profile:** A user profile is created based on the items they have interacted with in the past. The user profile contains the weighted importance of features based on their interactions.

- **Similarity Calculation:** The similarity between items or between items and the user profile is calculated using similarity metrics like cosine similarity or Euclidean distance.

- **Recommendation:** Items that are most similar to the user profile are recommended to the user.

Content-based recommendation systems are less affected by the cold-start problem as they can still recommend items based on their features. They are also more interpretable as they rely on item attributes. However, they may miss out on providing serendipitous recommendations and can be limited by the quality of feature extraction and user profiles.

**Choosing Between Collaborative Filtering and Content-Based:**
Both collaborative filtering and content-based approaches have their strengths and weaknesses. The choice between them depends on the specific requirements of the recommendation system, the type of data available, and the user base. Hybrid approaches that combine collaborative filtering and content-based techniques are also common, aiming to leverage the strengths of both methods and mitigate their weaknesses.

In this mini-project, you'll be building both content based and collaborative filtering engines for the [MovieLens 25M dataset](https://grouplens.org/datasets/movielens/25m/). The MovieLens 25M dataset is one of the most widely used and popular datasets for building and evaluating recommendation systems. It is provided by the GroupLens Research project, which collects and studies datasets related to movie ratings and recommendations. The MovieLens 25M dataset contains movie ratings and other related information contributed by users of the MovieLens website.

**Dataset Details:**
- **Size:** The dataset contains approximately 25 million movie ratings.
- **Users:** It includes ratings from over 162,000 users.
- **Movies:** The dataset consists of ratings for more than 62,000 movies.
- **Ratings:** The ratings are provided on a scale of 1 to 5, where 1 is the lowest rating and 5 is the highest.
- **Timestamps:** Each rating is associated with a timestamp, indicating when the rating was given.

**Data Files:**
The dataset is usually split into three CSV files:

1. **movies.csv:** Contains information about movies, including the movie ID, title, genres, and release year.
   - Columns: movieId, title, genres

2. **ratings.csv:** Contains movie ratings provided by users, including the user ID, movie ID, rating, and timestamp.
   - Columns: userId, movieId, rating, timestamp

3. **tags.csv:** Contains user-generated tags for movies, including the user ID, movie ID, tag, and timestamp.
   - Columns: userId, movieId, tag, timestamp

### Good Read: https://blog.dataiku.com/recommendation-engines-how-they-work-in-plain-english

First, import all the libraries you'll need.

In [1]:
import zipfile
import numpy as np
import pandas as pd
from urllib.request import urlretrieve
from sklearn.metrics.pairwise import cosine_similarity

Next, download the relevant components of the MoveLens dataset. Note, these instructions are roughly based on the colab [here](https://colab.research.google.com/github/google/eng-edu/blob/main/ml/recommendation-systems/recommendation-systems.ipynb?utm_source=ss-recommendation-systems&utm_campaign=colab-external&utm_medium=referral&utm_content=recommendation-systems#scrollTo=O3bcgduFo4s6).

In [2]:
### IT TAKES TOO LONG TO DOWNLOAD, Please download from my submission link OR uncomment and download to the same folder ###

print("Downloading movielens data...")
#urlretrieve('http://files.grouplens.org/datasets/movielens/ml-25m.zip', 'movielens.zip')
#zip_ref = zipfile.ZipFile('movielens.zip', 'r')
#zip_ref.extractall()

#print("Done. Dataset contains:")
#print(zip_ref.read('ml-25m/movies.csv'))
#print(zip_ref.read('ml-25m/ratings.csv'))
#print(zip_ref.read('ml-25m/tags.csv'))

''' INFO
# The movies file contains a binary feature for each genre.
#genre_cols = [
#    "genre_unknown", "Action", "Adventure", "Animation", "Children", "Comedy",
#    "Crime", "Documentary", "Drama", "Fantasy", "Film-Noir", "Horror",
#    "Musical", "Mystery", "Romance", "Sci-Fi", "Thriller", "War", "Western"
#]
'''

movies_df = pd.read_csv('ml-25m/movies.csv')

ratings_df = pd.read_csv('ml-25m/ratings.csv')

tags_df = pd.read_csv('ml-25m/tags.csv')

In [3]:
movies_df

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy
...,...,...,...
62418,209157,We (2018),Drama
62419,209159,Window of the Soul (2001),Documentary
62420,209163,Bad Poems (2018),Comedy|Drama
62421,209169,A Girl Thing (2001),(no genres listed)


In [4]:
ratings_df

,userId,movieId,rating,timestamp
0,1,296,5.0,1147880044
1,1,306,3.5,1147868817
2,1,307,5.0,1147868828
3,1,665,5.0,1147878820
4,1,899,3.5,1147868510
...,...,...,...,...
25000090,162541,50872,4.5,1240953372
25000091,162541,55768,2.5,1240951998
25000092,162541,56176,2.0,1240950697
25000093,162541,58559,4.0,1240953434


Before doing any kind of machine learning, it's always good to familiarize yourself with the datasets you'lll be working with.

Here are your tasks:

1. Spend some time familiarizing yourself with both the `movies` and `ratings` dataframes. How many unique user ids are present? How many unique movies are there?
2. Create a new dataframe that merges the `movies` and `ratings` tables on 'movie_id'. Only keep the 'user_id', 'title', 'rating' fields in this new dataframe.

In [5]:
# Spend some time familiarizing yourself with both the movies and ratings
# dataframes. How many unique user ids are present? How many unique movies
# are there?

print("Number of Unique Movies: ", len(movies_df['movieId'].unique()))

movies_df


Number of Unique Movies:  62423


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy
...,...,...,...
62418,209157,We (2018),Drama
62419,209159,Window of the Soul (2001),Documentary
62420,209163,Bad Poems (2018),Comedy|Drama
62421,209169,A Girl Thing (2001),(no genres listed)


In [6]:
print("Number of Unique User IDs: ", len(ratings_df['userId'].unique()))
print("Number of Unique Movies: ", len(ratings_df['movieId'].unique()))

ratings_df

Number of Unique User IDs:  162541
Number of Unique Movies:  59047


,userId,movieId,rating,timestamp
0,1,296,5.0,1147880044
1,1,306,3.5,1147868817
2,1,307,5.0,1147868828
3,1,665,5.0,1147878820
4,1,899,3.5,1147868510
...,...,...,...,...
25000090,162541,50872,4.5,1240953372
25000091,162541,55768,2.5,1240951998
25000092,162541,56176,2.0,1240950697
25000093,162541,58559,4.0,1240953434


In [7]:
# Merge movies and ratings dataframes

movies_ratings_df = movies_df.merge(ratings_df, on='movieId')
movies_ratings_df = movies_ratings_df.reindex(columns=['userId', 'movieId', 'title', 'genres', 'rating', 'timestamp'])
movies_ratings_df

,userId,movieId,title,genres,rating,timestamp
0,2,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,3.5,1141415820
1,3,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,4.0,1439472215
2,4,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,3.0,1573944252
3,5,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,4.0,858625949
4,8,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,4.0,890492517
...,...,...,...,...,...,...
25000090,119571,209157,We (2018),Drama,1.5,1574280748
25000091,115835,209159,Window of the Soul (2001),Documentary,3.0,1574280985
25000092,6964,209163,Bad Poems (2018),Comedy|Drama,4.5,1574284913
25000093,119571,209169,A Girl Thing (2001),(no genres listed),3.0,1574291826


In [8]:
df = movies_ratings_df[['userId', 'title', 'rating']]   # For Requirement Above
df

,userId,title,rating
0,2,Toy Story (1995),3.5
1,3,Toy Story (1995),4.0
2,4,Toy Story (1995),3.0
3,5,Toy Story (1995),4.0
4,8,Toy Story (1995),4.0
...,...,...,...
25000090,119571,We (2018),1.5
25000091,115835,Window of the Soul (2001),3.0
25000092,6964,Bad Poems (2018),4.5
25000093,119571,A Girl Thing (2001),3.0


In [9]:
df = movies_ratings_df[['movieId', 'userId', 'title', 'rating', 'genres']]   # For Content-Based Filtering in next section 
df

,movieId,userId,title,rating,genres
0,1,2,Toy Story (1995),3.5,Adventure|Animation|Children|Comedy|Fantasy
1,1,3,Toy Story (1995),4.0,Adventure|Animation|Children|Comedy|Fantasy
2,1,4,Toy Story (1995),3.0,Adventure|Animation|Children|Comedy|Fantasy
3,1,5,Toy Story (1995),4.0,Adventure|Animation|Children|Comedy|Fantasy
4,1,8,Toy Story (1995),4.0,Adventure|Animation|Children|Comedy|Fantasy
...,...,...,...,...,...
25000090,209157,119571,We (2018),1.5,Drama
25000091,209159,115835,Window of the Soul (2001),3.0,Documentary
25000092,209163,6964,Bad Poems (2018),4.5,Comedy|Drama
25000093,209169,119571,A Girl Thing (2001),3.0,(no genres listed)


As mentioned in the introduction, content-Based Filtering is a recommendation engine approach that focuses on the attributes or features of items (products, movies, music, articles, etc.) and leverages these features to make personalized recommendations. The underlying idea is to match the characteristics of items with the preferences of users to suggest items that align with their interests. Content-based filtering is particularly useful when explicit user-item interactions (e.g., ratings or purchases) are sparse or unavailable.

**Key Steps in Content-Based Filtering:**

1. **Feature Extraction:**
   - For each item, relevant features are extracted. These features are typically descriptive attributes that can be represented numerically, such as genre, director, actors, author, publication date, and keywords.
   - In the case of text-based items, natural language processing techniques may be used to extract features like TF-IDF (Term Frequency-Inverse Document Frequency) scores.

2. **User Profile Creation:**
   - A user profile is created based on the items they have interacted with in the past. The user profile contains the weighted importance of features based on their interactions.
   - For example, if a user has watched several action movies, the action genre feature would receive a higher weight in their profile.

3. **Similarity Calculation:**
   - The similarity between items or between items and the user profile is calculated using similarity metrics like cosine similarity, Euclidean distance, or Pearson correlation.
   - Cosine similarity is commonly used as it measures the cosine of the angle between two vectors, which represents their similarity.

4. **Recommendation:**
   - Items that are most similar to the user profile are recommended to the user. These are items whose features have the highest similarity scores with the user profile.
   - The recommended items are presented as a list sorted by their similarity scores.

**Advantages of Content-Based Filtering:**
1. **No Cold-Start Problem:** Content-based filtering can make recommendations even for new users with no historical interactions because it relies on item features rather than user history.

2. **User Independence:** The recommendations are based solely on the features of items and do not require knowledge of other users' preferences or behavior.

3. **Transparency:** Content-based recommendations are interpretable, as they depend on the features of items, making it easier for users to understand why specific items are recommended.

4. **Serendipity:** Content-based filtering can recommend items with characteristics not seen before by the user, leading to serendipitous discoveries.

5. **Diversity in Recommendations:** The method can offer diverse recommendations since it suggests items with different feature combinations.

**Limitations of Content-Based Filtering:**
1. **Limited Discovery:** Content-based filtering may struggle to recommend items outside the scope of users' historical interactions or interests.

2. **Over-Specialization:** Users may receive recommendations that are too similar to their previous choices, leading to a lack of exposure to new item categories.

3. **Dependency on Feature Quality:** The quality and relevance of item features significantly influence the quality of recommendations.

4. **Limited for Cold Items:** Content-based filtering can struggle to recommend new items with limited feature information.

Here is your task:

1. Write a function that takes in a user id and the dataframe you created before that contains 'user_id', 'title', and 'rating'. The function should return content-based recommendations for this user. Here are steps you can take:

  A. Get the user's rated movies

  B. Create a TF-IDF matrix using movie genres. Note, this can be extracted from the `movies` dataframe.

  C. Compute the cosine similarity between movie genres. Use the [cosine_similarity](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_similarity.html) function.

  D. Get the indices of similar movies to those rated by the user based on cosine similarity. Keep only the top 5.

  E. Remove duplicates and movies already rated by the user.

In [10]:
### TESTING WITHOUT FUNCTION ###

# Reference Help: 
# https://www.kaggle.com/code/ibtesama/getting-started-with-a-movie-recommendation-system
# https://www.codeheroku.com/post.html?name=Building%20a%20Movie%20Recommendation%20Engine%20in%20Python%20using%20Scikit-Learn
# https://github.com/pravinkumarosingh/MoRe , Same as Heroku


# Uncomment block to test
'''
from sklearn.feature_extraction.text import TfidfVectorizer

limit = 65000   # To Limit Processing Time for Cosine Similarity

# Get the user's rated movies
print("User Rated Movies: ")
user_rated_df = df[df['userId'] == 3]
user_rated_df = user_rated_df.sort_values(by=['rating'], ascending=False)
print(user_rated_df)

# Create a TF-IDF matrix using movie genres
print("TF-IDF Matrix: ")
tf = TfidfVectorizer(analyzer=lambda s: s.split('|'))
tfidf_matrix = tf.fit_transform(movies_df['genres'])
print(tfidf_matrix)

tfidf_matrix = tfidf_matrix[:limit]

movies_df = movies_df[:limit]
'''

'\nfrom sklearn.feature_extraction.text import TfidfVectorizer\n\nlimit = 65000   # To Limit Processing Time for Cosine Similarity\n\n# Get the user\'s rated movies\nprint("User Rated Movies: ")\nuser_rated_df = df[df[\'userId\'] == 3]\nuser_rated_df = user_rated_df.sort_values(by=[\'rating\'], ascending=False)\nprint(user_rated_df)\n\n# Create a TF-IDF matrix using movie genres\nprint("TF-IDF Matrix: ")\ntf = TfidfVectorizer(analyzer=lambda s: s.split(\'|\'))\ntfidf_matrix = tf.fit_transform(movies_df[\'genres\'])\nprint(tfidf_matrix)\n\ntfidf_matrix = tfidf_matrix[:limit]\n\nmovies_df = movies_df[:limit]\n'

In [11]:
# Uncomment block to test
'''
#genre_df = pd.DataFrame(tfidf_matrix.todense(), columns=tf.get_feature_names_out(), index=movies_df.title).sample(20, axis=1).sample(20, axis=0)   # Sampled
genre_df = pd.DataFrame(tfidf_matrix.todense(), columns=tf.get_feature_names_out(), index=movies_df.title)
genre_df
'''

'\n#genre_df = pd.DataFrame(tfidf_matrix.todense(), columns=tf.get_feature_names_out(), index=movies_df.title).sample(20, axis=1).sample(20, axis=0)   # Sampled\ngenre_df = pd.DataFrame(tfidf_matrix.todense(), columns=tf.get_feature_names_out(), index=movies_df.title)\ngenre_df\n'

In [12]:
# Takes 3-4 minutes to run

# Uncomment block to test
'''
cosine_sim = cosine_similarity(genre_df)
cosine_sim
'''

'\ncosine_sim = cosine_similarity(genre_df)\ncosine_sim\n'

In [13]:
# Test to get row index from title
#title = 'Dangerous Liaisons (1959)'
#print(movies_df.index[movies_df['title'] == title][0])


# Uncomment block to test
'''
rec_dict = {}
titles = user_rated_df['title'].tolist()[:5]
print(titles)
for i, title in enumerate(titles):
    movie_index = movies_df.index[movies_df['title'] == title][0]
    #print(movie_index)
    similar_titles = list(enumerate(cosine_sim[movie_index]))
    sorted_similar_titles = sorted(similar_titles, key = lambda x:x[1], reverse = True)
    #print(sorted_similar_titles)
    
    rec_titles = []
    for i, movie in enumerate(sorted_similar_titles):
        recommended_title_from_index = genre_df.index[movie[0]]
        if title == recommended_title_from_index:   # Account for duplicates
            print(f"Title and Recommended Title match --> Ignoring: {title} and {recommended_title_from_index}")
            continue
        rec_titles.append(recommended_title_from_index)
        if i == 5:   # Gets 6 titles
            if len(rec_titles) > 5:   # If 6 title, will remove the last title
                rec_titles = rec_titles[:-1]
            rec_dict[title] = rec_titles
            rec_titles = []
            break

for movie, movie_recommendation in rec_dict.items():
    print(f"Movie: {movie}, \n  Recommended Movies: {movie_recommendation}")
'''

'\nrec_dict = {}\ntitles = user_rated_df[\'title\'].tolist()[:5]\nprint(titles)\nfor i, title in enumerate(titles):\n    movie_index = movies_df.index[movies_df[\'title\'] == title][0]\n    #print(movie_index)\n    similar_titles = list(enumerate(cosine_sim[movie_index]))\n    sorted_similar_titles = sorted(similar_titles, key = lambda x:x[1], reverse = True)\n    #print(sorted_similar_titles)\n    \n    rec_titles = []\n    for i, movie in enumerate(sorted_similar_titles):\n        recommended_title_from_index = genre_df.index[movie[0]]\n        if title == recommended_title_from_index:   # Account for duplicates\n            print(f"Title and Recommended Title match --> Ignoring: {title} and {recommended_title_from_index}")\n            continue\n        rec_titles.append(recommended_title_from_index)\n        if i == 5:   # Gets 6 titles\n            if len(rec_titles) > 5:   # If 6 title, will remove the last title\n                rec_titles = rec_titles[:-1]\n            rec_dict

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer

limit = 65000   # Lower to Run Faster, Limit Processing Time for Cosine Similarity

# Content-Based Filtering using Movie Genres, input is the User ID
def content_based_recommendation_on_user(user_id, df):
    movies_df = pd.read_csv('ml-25m/movies.csv')

    # Get the user's rated movies
    print("User Rated Movies: ")
    user_rated_df = df[df['userId'] == user_id]
    user_rated_df = user_rated_df.sort_values(by=['rating'], ascending=False)
    print(user_rated_df)

    # Create a TF-IDF matrix using movie genres
    print("TF-IDF Matrix: ")
    tf = TfidfVectorizer(analyzer=lambda s: s.split('|'))
    tfidf_matrix = tf.fit_transform(movies_df['genres'])
    
    tfidf_matrix = tfidf_matrix[:limit]
    print(tfidf_matrix)

    movies_df = movies_df[:limit]
    print(movies_df)

    genre_df = pd.DataFrame(tfidf_matrix.todense(), columns=tf.get_feature_names_out(), index=movies_df.title)
    print(genre_df)

    """ ### Cosine Similarity Takes 3-4 Minutes to run ### """
    # Compute the cosine similarity between movie genres
    cosine_sim = cosine_similarity(genre_df)
    print(cosine_sim)

    # Get the indices of the similar movies based on cosine similarity
    rec_dict = {}
    titles = user_rated_df['title'].tolist()[:5]
    print("==================================================================================================================")
    print(f"\nTop five titles: {titles}")
    for i, title in enumerate(titles):
        movie_index = movies_df.index[movies_df['title'] == title][0]
        #print(movie_index)
        similar_titles_index_similarity = list(enumerate(cosine_sim[movie_index]))
        #print(similar_titles_index_similarity)
        sorted_similar_titles_index_similarity = sorted(similar_titles_index_similarity, key = lambda x:x[1], reverse = True)
        #print(sorted_similar_titles_index_similarity)
        
        rec_titles = []
        for i, sim_title_index_similarity in enumerate(sorted_similar_titles_index_similarity):
            recommended_title_from_index = genre_df.index[sim_title_index_similarity[0]]
            # Remove duplicates and movies already rated by the user
            if title == recommended_title_from_index:   # Account for duplicates
                print(f"\nTitle and Recommended Title match --> * Ignoring - Title: {title} and Recommended Title: {recommended_title_from_index} *")
                continue
            rec_titles.append(recommended_title_from_index)
            if i == 5:   # Gets 6 titles
                if len(rec_titles) > 5:   # If 6 title, will remove the last title
                    rec_titles = rec_titles[:-1]
                rec_dict[title] = rec_titles
                rec_titles = []
                break

    for movie, movie_recommendation in rec_dict.items():   # Outputs the top five movies the user has rated and recommends similar movies
        print(f"\nMovie: {movie}, \n  Recommended Movies: {movie_recommendation}")

#content_based_recommendation_on_user(3, df)

My Own Side Work - Do something similar as main task above using only a single movie as input instead of a user.

In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer

limit = 65000   # Lower to Run Faster, Limit Processing Time for Cosine Similarity

# Content-Based Filtering using Movie Genres, input is the Movie Title
def content_based_recommendation_on_movie(movie_title, df):
    movies_df = pd.read_csv('ml-25m/movies.csv')

    # Create a TF-IDF matrix using movie genres
    print("TF-IDF Matrix: ")
    tf = TfidfVectorizer(analyzer=lambda s: s.split('|'))
    tfidf_matrix = tf.fit_transform(movies_df['genres'])
    
    tfidf_matrix = tfidf_matrix[:limit]
    print(tfidf_matrix)

    movies_df = movies_df[:limit]
    print(movies_df)

    genre_df = pd.DataFrame(tfidf_matrix.todense(), columns=tf.get_feature_names_out(), index=movies_df.title)
    print(genre_df)

    """ ### Cosine Similarity Takes 3-4 Minutes to run ### """
    # Compute the cosine similarity between movie genres
    cosine_sim = cosine_similarity(genre_df)
    print(cosine_sim)

    # Get the indices of the similar movies based on cosine similarity
    rec_dict = {}
    title = movie_title
    print("==================================================================================================================")
    print(f"\nThe Movie Title: {title}")
    
    movie_index = movies_df.index[movies_df['title'] == title][0]
    #print(movie_index)
    similar_titles_index_similarity = list(enumerate(cosine_sim[movie_index]))
    #print(similar_titles_index_similarity)
    sorted_similar_titles_index_similarity = sorted(similar_titles_index_similarity, key = lambda x:x[1], reverse = True)
    #print(sorted_similar_titles_index_similarity)

    rec_titles = []
    for i, sim_title_index_similarity in enumerate(sorted_similar_titles_index_similarity):
        recommended_title_from_index = genre_df.index[sim_title_index_similarity[0]]
        # Remove duplicates and movies already rated by the user
        if title == recommended_title_from_index:   # Account for duplicates
            print(f"\nTitle and Recommended Title match --> * Ignoring - Title: {title} and Recommended Title: {recommended_title_from_index} *")
            continue
        rec_titles.append(recommended_title_from_index)
        if i == 10:   # Gets up to 11 titles
            if len(rec_titles) > 10:   # If 11 titles, will remove the last title
                rec_titles = rec_titles[:-1]
            rec_dict[title] = rec_titles
            rec_titles = []
            break

    for movie, movie_recommendation in rec_dict.items():   # Outputs the top five movies the user has rated and recommends similar movies
        print(f"\nMovie: {movie}, \n  Recommended Movies: {movie_recommendation}")

content_based_recommendation_on_movie("Pride and Prejudice (1940)", df)

TF-IDF Matrix: 
  (0, 9)	0.4967483702845257
  (0, 5)	0.27771718920269134
  (0, 4)	0.48808437174545455
  (0, 3)	0.48833048769293214
  (0, 2)	0.44656600888161224
  (1, 9)	0.6004535115193032
  (1, 4)	0.5899807477262311
  (1, 2)	0.5397946811673262
  (2, 15)	0.8011493881971549
  (2, 5)	0.5984644164788115
  (3, 8)	0.44022013245613556
  (3, 15)	0.7193439273049612
  (3, 5)	0.5373551425545094
  (4, 5)	1.0
  (5, 17)	0.5370772735955626
  (5, 6)	0.6249107985241872
  (5, 1)	0.5665990611314318
  (6, 15)	0.8011493881971549
  (6, 5)	0.5984644164788115
  (7, 4)	0.7377898038839786
  (7, 2)	0.6750305217431583
  (8, 1)	1.0
  (9, 17)	0.5220827968697404
  (9, 1)	0.5507803757155867
  (9, 2)	0.6512069801063765
  :	:
  (62407, 0)	1.0
  (62408, 7)	0.6368002861924501
  (62408, 9)	0.771028790321875
  (62409, 19)	1.0
  (62410, 8)	1.0
  (62411, 5)	1.0
  (62412, 7)	0.6432580604344854
  (62412, 3)	0.7656494417721884
  (62413, 8)	1.0
  (62414, 11)	0.6946781180206175
  (62414, 6)	0.7193207298162155
  (62415, 0)	1.0
  (

The key idea behind collaborative filtering is that users who have agreed in the past will likely agree in the future. Instead of relying on item attributes or user profiles, collaborative filtering identifies patterns of user behavior and item preferences from the interactions present in the data.

**Types of Collaborative Filtering:**
There are two main types of collaborative filtering:

**Collaborative Filtering Process:**
The collaborative filtering process typically involves the following steps:

1. **Data Collection:**
   - Gather data on user-item interactions, such as movie ratings, product purchases, or article clicks.

2. **User-Item Matrix:**
   - Organize the data into a user-item matrix, where rows represent users, columns represent items, and the entries contain the users' interactions (e.g., ratings).

3. **Similarity Calculation:**
   - Calculate the similarity between users or items using similarity metrics such as cosine similarity, Pearson correlation, or Jaccard similarity.
   - For user-based collaborative filtering, user similarities are calculated, and for item-based collaborative filtering, item similarities are calculated.

4. **Neighborhood Selection:**
   - For each user or item, select the most similar users or items as the neighborhood.
   - The size of the neighborhood (the number of similar users or items to consider) is an important parameter to control the system's behavior.

5. **Prediction Generation:**
   - Predict the ratings for items that the target user has not yet interacted with by combining the ratings of neighboring users or items.

6. **Recommendation Generation:**
   - Recommend items with the highest predicted ratings to the target user.

**Advantages of Collaborative Filtering using User-Item Interactions:**
- Collaborative filtering is based solely on user interactions and does not require knowledge of item attributes, making it useful for cases where item data is sparse or unavailable.
- It can provide serendipitous recommendations, suggesting items that users may not have discovered on their own.
- Collaborative filtering can be applied in various domains, including e-commerce, music, movie, and content recommendations.

**Limitations of Collaborative Filtering:**
- The cold-start problem: Collaborative filtering struggles to recommend to new users or items with no or limited interaction history.
- It may suffer from sparsity when data is limited or when users have only interacted with a small subset of items.
- Scalability issues can arise with large datasets and an increasing number of users or items.

Here is your task:

1. Write a function that takes in a user id and the dataframe you created before that contains 'user_id', 'title', and 'rating'. The function should return collaborative filtering recommendations for this user based on a user-item interaction matrix. Here are steps you can take:

  A. Create the user-item matrix using Pandas' [pivot_table](https://pandas.pydata.org/docs/reference/api/pandas.pivot_table.html).

  B. Fill missing values with zeros in this matrix.

  C. Calculate user-user similarity matrix using cosine similarity.

  D. Get the array of similarity scores of the target user with all other users from the similarity matrix.

  E. Extract, say the the top 5 most similar users (excluding the target user).

  F. Generate movie recommendations based on the most similar users.

  G. Remove duplicate movies recommendations.

In [16]:
### TESTING WITHOUT FUNCTION ###

# Reference Help: 
# https://www.kaggle.com/code/ibtesama/getting-started-with-a-movie-recommendation-system
# https://memgraph.com/blog/cosine-similarity-python-scikit-learn
# https://www.codeheroku.com/post.html?name=Building%20a%20Movie%20Recommendation%20Engine%20in%20Python%20using%20Scikit-Learn
# https://github.com/pravinkumarosingh/MoRe , Same as Heroku


# Uncomment block to test
'''
from sklearn.feature_extraction.text import CountVectorizer

movies_df = pd.read_csv('ml-25m/movies.csv')

# Get the user's rated movies
print("User Rated Movies: ")
#user_rated_df = df[df['userId'] == 3]
user_rated_df2 = df[['userId', 'title', 'rating', 'genres']]
user_rated_df2_sample = user_rated_df2.sample(n=25000, random_state=0)   # Dataset Too large To Create Pivot Table
print(user_rated_df2_sample)

# Create the user-item matrix
# Fill missing values with 0 (indicating no rating)
user_item_matrix = pd.pivot_table(user_rated_df2_sample, index='userId', columns=['title', 'genres'], values='rating', fill_value=0)
print(user_item_matrix)

# Calculate user-user similarity matrix using cosine similarity
user_rated_df2_sample.sort_values(by=['userId'], ascending=True)
#print(user_rated_df2_sample.sort_values(by=['userId'], ascending=True))

user_rated_df2_sample['rating'] = user_rated_df2_sample['rating'].astype(str)

user_rated_df2_sample = user_rated_df2_sample.groupby('userId').agg({'title':' '.join, 'rating':' '.join, 'genres':' '.join}).reset_index()
print(user_rated_df2_sample)

features = ['title', 'rating', 'genres']

def combine_features(row):
    return row['title'] + " " + str(row['rating']) + " " +row["genres"]

for feature in features:
    user_rated_df2_sample[feature] = user_rated_df2_sample[feature].fillna('')

user_rated_df2_sample["combined_features"] = user_rated_df2_sample.apply(combine_features,axis=1)
cv = CountVectorizer()
count_matrix = cv.fit_transform(user_rated_df2_sample["combined_features"])
cosine_sim2 = cosine_similarity(count_matrix)
print(cosine_sim2)

# Get the similarity scores of the target user with all other users
user_id_list = user_rated_df2_sample['userId'].values
print(user_id_list)
user_cos_list = cosine_sim2[0]
print(user_cos_list)
user_cos_dict = dict(zip(user_id_list, user_cos_list))
print(user_cos_dict)

# Find the top N most similar users (excluding the target user)

#test = np.sort(cosine_sim2[0])[::-1][0:6]   # Just Testing Sort Values, Reverse Order / Descending Order, 6 Items to ignore first

similar_users = sorted(user_cos_dict.items(), key=lambda val: val[1], reverse=True)

# Other Technique From Previous Section
#similar_titles_index_similarity = list(enumerate(cosine_sim[movie_index]))
#print(similar_titles_index_similarity)
#sorted_similar_titles_index_similarity = sorted(similar_titles_index_similarity, key = lambda x:x[1], reverse = True)
#print(sorted_similar_titles_index_similarity)

similar_users_list = [user[0] for user in similar_users[1:6][:]]
print(similar_users_list)

# Generate movie recommendations based on the most similar users
# # Remove duplicates from recommendations
content_based_recommendation_on_user(similar_users_list[0], df)
'''

'\nfrom sklearn.feature_extraction.text import CountVectorizer\n\nmovies_df = pd.read_csv(\'ml-25m/movies.csv\')\n\n# Get the user\'s rated movies\nprint("User Rated Movies: ")\n#user_rated_df = df[df[\'userId\'] == 3]\nuser_rated_df2 = df[[\'userId\', \'title\', \'rating\', \'genres\']]\nuser_rated_df2_sample = user_rated_df2.sample(n=25000, random_state=0)   # Dataset Too large To Create Pivot Table\nprint(user_rated_df2_sample)\n\n# Create the user-item matrix\n# Fill missing values with 0 (indicating no rating)\nuser_item_matrix = pd.pivot_table(user_rated_df2_sample, index=\'userId\', columns=[\'title\', \'genres\'], values=\'rating\', fill_value=0)\nprint(user_item_matrix)\n\n# Calculate user-user similarity matrix using cosine similarity\nuser_rated_df2_sample.sort_values(by=[\'userId\'], ascending=True)\n#print(user_rated_df2_sample.sort_values(by=[\'userId\'], ascending=True))\n\nuser_rated_df2_sample[\'rating\'] = user_rated_df2_sample[\'rating\'].astype(str)\n\nuser_rated_df

In [17]:
from sklearn.feature_extraction.text import CountVectorizer

# Collaborative Filtering using User-Item Interactions
def collaborative_filtering_recommendation(user_id, df):
    movies_df = pd.read_csv('ml-25m/movies.csv')

    # Get the user's rated movies
    print("User Rated Movies: ")
    #user_rated_df = df[df['userId'] == 3]
    user_rated_df2 = df[['userId', 'title', 'rating', 'genres']]
    user_rated_df2_sample = user_rated_df2.sample(n=25000, random_state=0)   # Dataset Too large To Create Pivot Table
    print(user_rated_df2_sample)

    # Create the user-item matrix
    # Fill missing values with 0 (indicating no rating)
    user_item_matrix = pd.pivot_table(user_rated_df2_sample, index='userId', columns=['title', 'genres'], values='rating', fill_value=0)
    print(user_item_matrix)

    # Calculate user-user similarity matrix using cosine similarity
    user_rated_df2_sample.sort_values(by=['userId'], ascending=True)
    #print(user_rated_df2_sample.sort_values(by=['userId'], ascending=True))

    user_rated_df2_sample['rating'] = user_rated_df2_sample['rating'].astype(str)

    user_rated_df2_sample = user_rated_df2_sample.groupby('userId').agg({'title':' '.join, 'rating':' '.join, 'genres':' '.join}).reset_index()
    print(user_rated_df2_sample)

    features = ['title', 'rating', 'genres']

    def combine_features(row):
        return row['title'] + " " + str(row['rating']) + " " +row["genres"]

    for feature in features:
        user_rated_df2_sample[feature] = user_rated_df2_sample[feature].fillna('')

    user_rated_df2_sample["combined_features"] = user_rated_df2_sample.apply(combine_features,axis=1)
    cv = CountVectorizer()
    count_matrix = cv.fit_transform(user_rated_df2_sample["combined_features"])
    cosine_sim2 = cosine_similarity(count_matrix)
    print(cosine_sim2)

    # Get the similarity scores of the target user with all other users
    user_id_list = user_rated_df2_sample['userId'].values
    print(user_id_list)
    user_cos_list = cosine_sim2[0]
    print(user_cos_list)
    user_cos_dict = dict(zip(user_id_list, user_cos_list))
    print(user_cos_dict)

    # Find the top N most similar users (excluding the target user)

    #test = np.sort(cosine_sim2[0])[::-1][0:6]   # Just Testing Sort Values, Reverse Order / Descending Order, 6 Items to ignore first

    similar_users = sorted(user_cos_dict.items(), key=lambda val: val[1], reverse=True)
    ''' Other Technique From Previous Section
    similar_titles_index_similarity = list(enumerate(cosine_sim[movie_index]))
    #print(similar_titles_index_similarity)
    sorted_similar_titles_index_similarity = sorted(similar_titles_index_similarity, key = lambda x:x[1], reverse = True)
    #print(sorted_similar_titles_index_similarity)
    '''
    similar_users_list = [user[0] for user in similar_users[1:6][:]]
    print(similar_users_list)
    return similar_users_list

# Generate movie recommendations based on the most similar users
# # Remove duplicates from recommendations
collaborative_filtering_recommendation(12, df)   # '12' is a valid user in the sample, takes too long to run if using the whole dataset
content_based_recommendation_on_user(similar_users_list[0], df)

Now, test your recommendations engines! Select a few user ids and generate recommendations using both functions you've written. Are the recommendations similar? Do the recommendations make sense?

In [18]:
# Test the recommendation engines

# WILL TAKE 3 - 4 MINUTES TO RUN ####
#similar_users_list = collaborative_filtering_recommendation(12, df)   # '12' is a valid user in the sample, takes too long to run if using the whole dataset
#print(f"Similar Users within the sample: {similar_users_list}")
#content_based_recommendation_on_user(similar_users_list[0], df)